In [7]:
import pandas as pd
from pathlib import Path 

In [8]:
current_dir = Path(Path.cwd()).parent
data_dir = current_dir / "data"
print(data_dir)

d:\pedro\Programacao\.Projetos\dengue_prediction\dengue_prediction\data


In [9]:

import re
import unicodedata
 
def show_columns(df, n_range = 7):
    df = list(df.columns)
    l = []
    range = n_range
    for c in df:
        if range == n_range:
            print(l)
            l = []
            range = 0
        range += 1
        l.append(c)
    print("\n")

def normalize_column(col):
    # tira acentos
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    # minúsculas
    col = col.lower()
    # troca qualquer coisa que não seja letra ou número por _
    col = re.sub(r"[^a-z0-9]+", "_", col)
    # remove _ no começo e no fim
    col = col.strip("_")
    return col

### casos-de-dengue data

In [10]:
dengue_dir = data_dir / "dengue_data"
cs_dengue_2013 = pd.read_csv(dengue_dir / "casos-de-dengue-em-2013.csv", sep=";")

cs_dengue_2013.columns = [normalize_column(col) for col in cs_dengue_2013.columns]
show_columns(df=cs_dengue_2013)
cs_dengue_2013.head()

[]
['nu_notificacao', 'tp_notificacao', 'co_cid', 'dt_notificacao', 'ds_semana_notificacao', 'notificacao_ano', 'co_uf_notificacao']
['co_municipio_notificacao', 'id_regional', 'co_unidade_notificacao', 'dt_diagnostico_sintoma', 'ds_semana_sintoma', 'dt_nascimento', 'nu_idade']
['tp_sexo', 'tp_gestante', 'tp_raca_cor', 'tp_escolaridade', 'co_uf_residencia', 'co_municipio_residencia', 'co_regional_residencia']
['co_distrito_residencia', 'co_bairro_residencia', 'no_bairro_residencia', 'co_logradouro_residencia', 'nome_logradouro_residencia', 'co_geo_campo_1', 'co_geo_campo_2']
['ds_referencia_residencia', 'nu_cep_residencia', 'tp_zona_residencia', 'co_pais_residencia', 'tp_duplicidade', 'dt_digitacao', 'dt_transf_us']
['dt_transf_dm', 'dt_transf_sm', 'dt_transf_rm', 'dt_transf_rs', 'dt_transf_se', 'nu_lote_vertical', 'nu_lote_horizontal']
['tp_fluxo_retorno', 'st_fluxo_retorno_recebido', 'ds_identificador_registro', 'st_importado', 'dt_investigacao', 'co_cbo_ocupacao', 'dt_coleta_exame']

,nu_notificacao,tp_notificacao,co_cid,dt_notificacao,ds_semana_notificacao,notificacao_ano,co_uf_notificacao,co_municipio_notificacao,id_regional,co_unidade_notificacao,...,tp_evolucao_caso,dt_obito,dt_encerramento,st_ocorreu_hospitalizacao,dt_internacao,co_uf_hospital,co_municipio_hospital,co_unidade_hospital,nu_ddd_hospital,nu_telefone_hospital
0,1224587,2,A90,2013/01/04 00:00:00,201301,2013,26,261160,1497.0,3008002,...,1.0,NaN,2013/02/20 00:00:00,2.0,NaN,NaN,NaN,NaN,NaN,2013008.0
1,1411256,2,A90,2013/01/07 00:00:00,201302,2013,26,261160,1497.0,5540739,...,1.0,NaN,2013/02/18 00:00:00,9.0,NaN,NaN,NaN,NaN,NaN,2013008.0
2,1418852,2,A90,2013/02/20 00:00:00,201308,2013,26,261160,1497.0,6481876,...,1.0,NaN,2013/03/29 00:00:00,9.0,NaN,NaN,NaN,NaN,NaN,2013018.0
3,1292875,2,A90,2013/01/03 00:00:00,201301,2013,26,261160,1497.0,2517140,...,1.0,NaN,2013/01/25 00:00:00,2.0,NaN,NaN,NaN,NaN,NaN,2013004.0
4,1416160,2,A90,2013/01/07 00:00:00,201302,2013,26,261160,1497.0,3008002,...,1.0,NaN,2013/01/10 00:00:00,9.0,NaN,NaN,NaN,NaN,NaN,2013005.0


In [11]:
# get only columns that we need (telefone -> municipio)
# Maybe would be interessinting other columns, but for now we will focus on these three
import datetime


# get only columns that we need (telefone -> municipio)
# Maybe would be interessinting other columns, but for now we will focus on these three

# rename
cs_dengue_2013 = cs_dengue_2013.rename(columns={
    "nu_notificacao": "id",
    "dt_notificacao": "data",
    "nu_telefone_hospital": "telefone_hospital"
})
cs_dengue_2013 = cs_dengue_2013[["id", "data", "telefone_hospital"]]

# data type
cs_dengue_2013["data"] = pd.to_datetime(
    cs_dengue_2013["data"],
    format="%Y/%m/%d %H:%M:%S",
    errors="coerce"
)
# verify if time is always same hour
if cs_dengue_2013["data"].dt.time.unique() ==  [datetime.time(0, 0)]:
    cs_dengue_2013["data"] = cs_dengue_2013["data"].dt.date

# remove duplicate 
cs_dengue_2013 = cs_dengue_2013.drop_duplicates(["id"])


print(cs_dengue_2013.info())
cs_dengue_2013.head()
   

<class 'pandas.core.frame.DataFrame'>
Index: 3187 entries, 0 to 3228
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 3187 non-null   int64  
 1   data               3187 non-null   object 
 2   telefone_hospital  3184 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 99.6+ KB
None


,id,data,telefone_hospital
0,1224587,2013-01-04,2013008.0
1,1411256,2013-01-07,2013008.0
2,1418852,2013-02-20,2013018.0
3,1292875,2013-01-03,2013004.0
4,1416160,2013-01-07,2013005.0


### Dados meteriologicos

In [12]:
weather_dir = data_dir / "weather_data"
weather_dir_2013 = weather_dir / "2013" / "INMET_NE_PE_A301_RECIFE_01-01-2013_A_31-12-2013.csv"
weather_2013 = pd.read_csv(weather_dir_2013, sep=";", encoding="latin1", skiprows=8, decimal=",")
weather_2013 = weather_2013.dropna(axis=1, how="all")

weather_2013.columns = [normalize_column(col) for col in weather_2013.columns]
show_columns(df=weather_2013, n_range=3)
weather_2013.head()

[]
['data_yyyy_mm_dd', 'hora_utc', 'precipitacao_total_horario_mm']
['pressao_atmosferica_ao_nivel_da_estacao_horaria_mb', 'pressao_atmosferica_max_na_hora_ant_aut_mb', 'pressao_atmosferica_min_na_hora_ant_aut_mb']
['radiacao_global_kj_m2', 'temperatura_do_ar_bulbo_seco_horaria_c', 'temperatura_do_ponto_de_orvalho_c']
['temperatura_maxima_na_hora_ant_aut_c', 'temperatura_minima_na_hora_ant_aut_c', 'temperatura_orvalho_max_na_hora_ant_aut_c']
['temperatura_orvalho_min_na_hora_ant_aut_c', 'umidade_rel_max_na_hora_ant_aut', 'umidade_rel_min_na_hora_ant_aut']
['umidade_relativa_do_ar_horaria', 'vento_direcao_horaria_gr_gr', 'vento_rajada_maxima_m_s']




,data_yyyy_mm_dd,hora_utc,precipitacao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,radiacao_global_kj_m2,temperatura_do_ar_bulbo_seco_horaria_c,temperatura_do_ponto_de_orvalho_c,temperatura_maxima_na_hora_ant_aut_c,temperatura_minima_na_hora_ant_aut_c,temperatura_orvalho_max_na_hora_ant_aut_c,temperatura_orvalho_min_na_hora_ant_aut_c,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria,vento_direcao_horaria_gr_gr,vento_rajada_maxima_m_s,vento_velocidade_horaria_m_s
0,2013-01-01,00:00,0.0,1012.4,1012.4,1012.1,-9999.0,26.5,22.1,26.7,26.5,22.2,21.2,77,72,77,155,4.5,1.6
1,2013-01-01,01:00,0.0,1012.4,1012.5,1012.3,-9999.0,26.4,20.9,26.7,26.4,22.1,20.7,77,70,72,156,4.7,1.5
2,2013-01-01,02:00,0.0,1012.1,1012.5,1012.1,-9999.0,26.3,21.9,26.5,26.3,21.9,21.0,76,72,76,158,4.7,1.8
3,2013-01-01,03:00,0.6,1011.5,1012.1,1011.5,-9999.0,25.1,22.5,26.4,24.8,22.6,21.9,86,76,86,120,5.3,2.0
4,2013-01-01,04:00,0.0,1010.9,1011.5,1010.9,-9999.0,25.4,22.0,25.6,25.0,22.5,21.9,86,81,82,118,4.5,1.5


In [13]:
weather_2013 = weather_2013.rename(columns={
    "data_yyyy_mm_dd": "data",
    "hora_utc": "hora",
})

# data type (verify)
cs_dengue_2013["data"] = pd.to_datetime(
    cs_dengue_2013["data"],
    format="%Y-%m-%d",
    errors="coerce"
)

print(weather_2013.info())
weather_2013.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 19 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   data                                                8760 non-null   object 
 1   hora                                                8760 non-null   object 
 2   precipitacao_total_horario_mm                       8760 non-null   float64
 3   pressao_atmosferica_ao_nivel_da_estacao_horaria_mb  8760 non-null   float64
 4   pressao_atmosferica_max_na_hora_ant_aut_mb          8760 non-null   float64
 5   pressao_atmosferica_min_na_hora_ant_aut_mb          8760 non-null   float64
 6   radiacao_global_kj_m2                               8760 non-null   float64
 7   temperatura_do_ar_bulbo_seco_horaria_c              8760 non-null   float64
 8   temperatura_do_ponto_de_orvalho_c                   8760 non-null   float64
 9

,data,hora,precipitacao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,radiacao_global_kj_m2,temperatura_do_ar_bulbo_seco_horaria_c,temperatura_do_ponto_de_orvalho_c,temperatura_maxima_na_hora_ant_aut_c,temperatura_minima_na_hora_ant_aut_c,temperatura_orvalho_max_na_hora_ant_aut_c,temperatura_orvalho_min_na_hora_ant_aut_c,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria,vento_direcao_horaria_gr_gr,vento_rajada_maxima_m_s,vento_velocidade_horaria_m_s
0,2013-01-01,00:00,0.0,1012.4,1012.4,1012.1,-9999.0,26.5,22.1,26.7,26.5,22.2,21.2,77,72,77,155,4.5,1.6
1,2013-01-01,01:00,0.0,1012.4,1012.5,1012.3,-9999.0,26.4,20.9,26.7,26.4,22.1,20.7,77,70,72,156,4.7,1.5
2,2013-01-01,02:00,0.0,1012.1,1012.5,1012.1,-9999.0,26.3,21.9,26.5,26.3,21.9,21.0,76,72,76,158,4.7,1.8
3,2013-01-01,03:00,0.6,1011.5,1012.1,1011.5,-9999.0,25.1,22.5,26.4,24.8,22.6,21.9,86,76,86,120,5.3,2.0
4,2013-01-01,04:00,0.0,1010.9,1011.5,1010.9,-9999.0,25.4,22.0,25.6,25.0,22.5,21.9,86,81,82,118,4.5,1.5
